## *The API key has been removed from the submission for security reasons. Please configure the environment variable OPENELECTRICITY_API_KEY before execution.

In [1]:
import requests
import pandas as pd

In [2]:
import os
# Set your OpenElectricity API key as an environment variable
# before running the code
# Example:
# os.environ["OPENELECTRICITY_API_KEY"] = "your_api_key" 
os.environ["OPENELECTRICITY_API_KEY"] = "your_api_key" # 临时设置环境变量（不要提交真实key）
API_KEY = os.getenv("OPENELECTRICITY_API_KEY")

## 1. per-facility power generated (5 min interval)

In [3]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

url = "https://api.openelectricity.org.au/v4/facilities/"

params = {
    "with_clerk": "true",
    "network_id": "NEM"
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)

data = response.json()

print(data.keys())

print(str(data)[:1000])

200
dict_keys(['version', 'created_at', 'success', 'data', 'total_records'])
{'version': '4.5.0.dev43', 'created_at': '2026-05-12T20:44:52+10:00', 'success': True, 'data': [{'code': 'ADP', 'code_display': 'ADP', 'name': 'Adelaide Desalination', 'network_id': 'NEM', 'network_region': 'SA1', 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>', 'osm_way_id': '12798948', 'location': {'lat': -35.100751, 'lng': 138.483357}, 'units': [{'code': 'ADPPV1', 'code_display': 'ADPPV1', 'fueltech_id': 'solar_utility', 'status_id': 'operating', 'capacity_registered': 24.75, 'capacity_maximum': 19.0, 'data_first_seen': '2021-05-18T13:10:00+10:00', 'data_last_seen': '2026-05-12T18:15:00+10:00', 'dispatch_type': 'GENERATOR', 'commencement_date': '2021-05-17

In [4]:
facilities_df = pd.DataFrame(data["data"])
print(len(facilities_df))

538


In [5]:
print(facilities_df[["code", "name"]].head(10))

       code                   name
0       ADP  Adelaide Desalination
1   ALDGASF                 Aldoga
2   AMCORGR            Amcor Glass
3  ANGASTON               Angaston
4       APS               Anglesea
5     APPIN                  Appin
6      ARWF                 Ararat
7     AVLSF                Avonlie
8  AWABAREF                  Awaba
9    DEIBDL             Bairnsdale


In [6]:
#调试阶段，先选3个code，正式时选5-10个
#power data

headers = {
    "Authorization": f"Bearer {API_KEY}"
}

facility_codes = [
    "APPIN",      # gas
    "DEIBDL",     # gas
    "BARCALDN",   # gas
    "BAYSW",      # coal
    "ANGASTON",   # distillate

    "ADP",        # battery/renewable
    "APS",        # coal_brown
    "ARWF",       # wind
    "AWABAREF",   # bioenergy
    "BANKSPT"     # distillate
]

url = "https://api.openelectricity.org.au/v4/data/facilities/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "power"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08")
]

# 添加多个 facility_code
for code in facility_codes:
    params.append(("facility_code", code))

response = requests.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)

data = response.json()

print(data.keys())

200
dict_keys(['version', 'created_at', 'success', 'data'])


In [7]:
print(data["data"][0].keys())

dict_keys(['network_code', 'metric', 'unit', 'interval', 'date_start', 'date_end', 'groupings', 'results', 'network_timezone_offset'])


In [8]:
results = data["data"][0]["results"]

print(type(results))
print(len(results))

<class 'list'>
13


In [9]:
first = results[0]

print(first.keys())

dict_keys(['name', 'date_start', 'date_end', 'columns', 'data'])


In [10]:
print(results[0]["columns"])
print(results[0]["data"][0])

{'unit_code': 'ADPBA1'}
['2026-05-01T00:00:00+10:00', -0.011]


In [11]:
rows = []

for r in results:

    unit_code = r["columns"]["unit_code"]

    for item in r["data"]:

        rows.append({
            "unit_code": unit_code,
            "timestamp": item[0],
            "power": item[1]
        })

power_df = pd.DataFrame(rows)

power_df["timestamp"] = pd.to_datetime(
    power_df["timestamp"],
    utc=True
)

power_df.head()

,unit_code,timestamp,power
0,ADPBA1,2026-04-30 14:00:00+00:00,-0.011
1,ADPBA1,2026-04-30 14:05:00+00:00,-0.021
2,ADPBA1,2026-04-30 14:10:00+00:00,0.106
3,ADPBA1,2026-04-30 14:15:00+00:00,-0.006
4,ADPBA1,2026-04-30 14:20:00+00:00,-0.037


## 2. per-facility CO2 emissions (5 min interval)

In [12]:
fossil_keywords = ["coal", "gas", "diesel", "distillate"]

rows = []

for _, row in facilities_df.iterrows():
    units = row["units"]

    for unit in units:
        fueltech = unit.get("fueltech_id", "")

        if any(k in fueltech for k in fossil_keywords):
            rows.append({
                "facility_code": row["code"],
                "facility_name": row["name"],
                "network_region": row["network_region"],
                "unit_code": unit.get("code"),
                "fueltech_id": fueltech,
                "status_id": unit.get("status_id")
            })

fossil_df = pd.DataFrame(rows)

fossil_df.head(20)

,facility_code,facility_name,network_region,unit_code,fueltech_id,status_id
0,AMCORGR,Amcor Glass,SA1,AMCORGR,distillate,retired
1,ANGASTON,Angaston,SA1,ANGAST1,distillate,operating
2,ANGASTON,Angaston,SA1,ANGAS1,distillate,retired
3,ANGASTON,Angaston,SA1,ANGAS2,distillate,retired
4,APS,Anglesea,VIC1,APS,coal_brown,retired
5,APPIN,Appin,NSW1,APPIN,gas_wcmg,operating
6,AWABAREF,Awaba,NSW1,AWABAREF,bioenergy_biogas,retired
7,DEIBDL,Bairnsdale,VIC1,BDL01,gas_ocgt,operating
8,DEIBDL,Bairnsdale,VIC1,BDL02,gas_ocgt,operating
9,BBASEHOS,Ballarat Hospital,VIC1,BBASEHOS,gas_recip,retired


In [13]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

facility_codes = [
    "APPIN",      # gas
    "DEIBDL",     # gas
    "BARCALDN",   # gas
    "BAYSW",      # coal
    "ANGASTON",   # distillate

    "ADP",        # battery/renewable
    "APS",        # coal_brown
    "ARWF",       # wind
    "AWABAREF",   # bioenergy
    "BANKSPT"     # distillate
]

url = "https://api.openelectricity.org.au/v4/data/facilities/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "emissions"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08")
]

for code in facility_codes:
    params.append(("facility_code", code))

response = requests.get(url, headers=headers, params=params)

print("Status code:", response.status_code)

data = response.json()

if response.status_code != 200 or not data.get("success", False):
    print(data)
else:
    results = data["data"][0]["results"]

    rows = []

    for r in results:
        unit_code = r["columns"].get("unit_code", None)

        for item in r["data"]:
            rows.append({
                "unit_code": unit_code,
                "timestamp": item[0],
                "emissions": item[1]
            })

    emissions_df = pd.DataFrame(rows)

    emissions_df["timestamp"] = pd.to_datetime(
        emissions_df["timestamp"],
        utc=True
    )

    emissions_df = emissions_df.drop_duplicates()
    emissions_df = emissions_df.dropna(subset=["unit_code", "timestamp"])

    print(emissions_df.head())
    print(emissions_df.shape)

Status code: 200
  unit_code                 timestamp  emissions
0    ADPBA1 2026-04-30 14:00:00+00:00        0.0
1    ADPBA1 2026-04-30 14:05:00+00:00        0.0
2    ADPBA1 2026-04-30 14:10:00+00:00        0.0
3    ADPBA1 2026-04-30 14:15:00+00:00        0.0
4    ADPBA1 2026-04-30 14:20:00+00:00        0.0
(24192, 3)


## 3. per-market price and demand (5 min interval)

In [14]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

url = "https://api.openelectricity.org.au/v4/market/network/NEM"

params = [
    ("interval", "5m"),
    ("metrics", "price"),
    ("metrics", "demand"),
    ("date_start", "2026-05-01"),
    ("date_end", "2026-05-08"),
    ("primary_grouping", "network_region")
]

response = requests.get(url, headers=headers, params=params)

print("Status code:", response.status_code)

data = response.json()
print(data.keys())
print(str(data)[:1000])

Status code: 200
dict_keys(['version', 'created_at', 'success', 'data'])
{'version': '4.5.0.dev43', 'created_at': '2026-05-12T20:44:55+10:00', 'success': True, 'data': [{'network_code': 'NEM', 'metric': 'price', 'unit': '$/MWh', 'interval': '5m', 'date_start': '2026-05-01T00:00:00+10:00', 'date_end': '2026-05-07T23:55:00+10:00', 'groupings': [], 'results': [{'name': 'price_NSW1', 'date_start': '2026-05-01T00:00:00+10:00', 'date_end': '2026-05-07T23:55:00+10:00', 'columns': {'region': 'NSW1'}, 'data': [['2026-05-01T00:00:00+10:00', 57.51], ['2026-05-01T00:05:00+10:00', 57.51], ['2026-05-01T00:10:00+10:00', 65.89], ['2026-05-01T00:15:00+10:00', 65.22], ['2026-05-01T00:20:00+10:00', 57.43], ['2026-05-01T00:25:00+10:00', 57.51], ['2026-05-01T00:30:00+10:00', 56.06], ['2026-05-01T00:35:00+10:00', 56.06], ['2026-05-01T00:40:00+10:00', 57.51], ['2026-05-01T00:45:00+10:00', 57.51], ['2026-05-01T00:50:00+10:00', 56.06], ['2026-05-01T00:55:00+10:00', 56.06], ['2026-05-01T01:00:00+10:00', 56.06],

In [15]:
for block in data["data"]:
    print("metric:", block["metric"])
    for r in block["results"]:
        print(r["name"], r["columns"])

metric: price
price_NSW1 {'region': 'NSW1'}
price_QLD1 {'region': 'QLD1'}
price_SA1 {'region': 'SA1'}
price_TAS1 {'region': 'TAS1'}
price_VIC1 {'region': 'VIC1'}
price_WEM {'region': 'WEM'}
metric: demand
demand_NSW1 {'region': 'NSW1'}
demand_QLD1 {'region': 'QLD1'}
demand_SA1 {'region': 'SA1'}
demand_TAS1 {'region': 'TAS1'}
demand_VIC1 {'region': 'VIC1'}
demand_WEM {'region': 'WEM'}


In [16]:
rows = []

for block in data["data"]:
    metric = block["metric"]

    for r in block["results"]:
        region = r["columns"].get("region")

        for item in r["data"]:
            rows.append({
                "region": region,
                "timestamp": item[0],
                metric: item[1]
            })

market_long_df = pd.DataFrame(rows)
market_long_df["timestamp"] = pd.to_datetime(
    market_long_df["timestamp"],
    utc=True
)

market_df = (
    market_long_df
    .groupby(["region", "timestamp"], as_index=False)
    .first()
)

market_df.head()

,region,timestamp,price,demand
0,NSW1,2026-04-30 14:00:00+00:00,57.51,7332.50
1,NSW1,2026-04-30 14:05:00+00:00,57.51,7323.78
2,NSW1,2026-04-30 14:10:00+00:00,65.89,7441.25
3,NSW1,2026-04-30 14:15:00+00:00,65.22,7417.92
4,NSW1,2026-04-30 14:20:00+00:00,57.43,7387.76


# Task 2：Data Integration and Materialisation/Caching

In [17]:
# 2.1 merge power and emissions
combined_df = pd.merge(
    power_df,
    emissions_df,
    on=["unit_code", "timestamp"],
    how="outer"
)

In [18]:
# 2.2 cleaning
combined_df["timestamp"] = pd.to_datetime(
    combined_df["timestamp"],
    utc=True
)

combined_df["power"] = pd.to_numeric(
    combined_df["power"],
    errors="coerce"
)

combined_df["emissions"] = pd.to_numeric(
    combined_df["emissions"],
    errors="coerce"
)

combined_df = combined_df.drop_duplicates()

combined_df = combined_df.dropna(
    subset=["unit_code", "timestamp"]
)

combined_df = combined_df.sort_values(
    by=["timestamp", "unit_code"]
).reset_index(drop=True)

combined_df["power"] = combined_df["power"].round(2)  #保留小数点后两位

combined_df["emissions"] = combined_df["emissions"].round(2)

In [19]:
# 2.3 save
combined_df.to_csv(
    "combined_power_emissions.csv",
    index=False
)

print("combined_power_emissions.csv saved")

combined_power_emissions.csv saved


In [20]:
# 2.4 cleaning market and save
market_df["timestamp"] = pd.to_datetime(market_df["timestamp"])

market_df["price"] = pd.to_numeric(market_df["price"], errors="coerce")
market_df["demand"] = pd.to_numeric(market_df["demand"], errors="coerce")

market_df = market_df.drop_duplicates()
market_df = market_df[
    market_df["region"] != "WEM"         #only NEM
]
market_df = market_df.dropna(
    subset=["region", "timestamp"]
)

market_df = market_df.sort_values(
    by=["timestamp", "region"]
).reset_index(drop=True)

market_df["price"] = market_df["price"].round(2)

market_df["demand"] = market_df["demand"].round(2)

market_df.to_csv(
    "market_price_demand.csv",
    index=False
)

print("market_price_demand.csv saved")
print(market_df.head())

market_price_demand.csv saved
  region                 timestamp  price   demand
0   NSW1 2026-04-30 14:00:00+00:00  57.51  7332.50
1   QLD1 2026-04-30 14:00:00+00:00  64.65  6165.20
2    SA1 2026-04-30 14:00:00+00:00  -2.01  1507.69
3   TAS1 2026-04-30 14:00:00+00:00  78.95   944.41
4   VIC1 2026-04-30 14:00:00+00:00   4.55  4568.84


# Task 3: Data Publishing via MQTT

This section uses the cached CSV files generated in Task 2:

- `combined_power_emissions.csv`
- optional `market_price_demand.csv`

It publishes one JSON message per facility/unit power + emissions record in chronological event-time order. Each message contains the fields needed by the Task 5 dashboard.

**Interface for the dashboard teammate**

- MQTT broker host: `localhost` by default
- MQTT broker port: `1883` by default
- MQTT topic: `openelectricity/nem/facility_power_emissions`
- Main ID field: `facility_id`
- Also included: `unit_code` and, when metadata is available, `facility_code`
- Power unit: MW
- Emissions unit: tCO2e
- Timestamp format: ISO-8601 string with timezone, for example `2026-05-01T00:00:00+10:00`

If the dashboard database uses Assignment 1 facility IDs, set `FACILITY_ID_MODE = "facility_code"` only if the `facility_code` values match that database. Otherwise keep `FACILITY_ID_MODE = "unit_code"` and let the dashboard match on `unit_code`.

## 3.1 MQTT configuration

Set the MQTT broker, topic, CSV paths, delay between messages, and the `facility_id` matching rule used by the dashboard.

In [21]:
# Task 3/6 configuration
import os
import json
import time
import uuid
from pathlib import Path

import pandas as pd

try:
    import paho.mqtt.client as mqtt
except ImportError:
    mqtt = None
    print("paho-mqtt is not installed yet. Dry-run mode still works; install requirements.txt before real MQTT publishing.")


# ---------------------------------------------------------------------
# MQTT interface agreed with the dashboard teammate
# ---------------------------------------------------------------------
MQTT_BROKER_HOST = os.getenv("MQTT_BROKER_HOST", "localhost")
MQTT_BROKER_PORT = int(os.getenv("MQTT_BROKER_PORT", "1883"))
MQTT_TOPIC = os.getenv("MQTT_TOPIC", "openelectricity/nem/facility_power_emissions")

# Leave these as None when the local broker does not require authentication.
MQTT_USERNAME = os.getenv("MQTT_USERNAME") or None
MQTT_PASSWORD = os.getenv("MQTT_PASSWORD") or None

# CSV files generated by Task 2 above
COMBINED_CSV = Path("combined_power_emissions.csv")
MARKET_CSV = Path("market_price_demand.csv")

# Required by the assignment/rubric
PUBLISH_DELAY_SECONDS = 0.1
REPLAY_RESTART_DELAY_SECONDS = 60

# Use "unit_code" if the dashboard database is unit-level.
# Use "facility_code" if your Assignment 1 / Task 4 table uses OpenElectricity facility codes.
FACILITY_ID_MODE = os.getenv("FACILITY_ID_MODE", "unit_code").lower()

# For maximum Task 6 efficiency, set this to True.
# If the marker/dashboard team wants every 5-minute row regardless of value change, set it to False.
PUBLISH_ONLY_CHANGED_VALUES = os.getenv("PUBLISH_ONLY_CHANGED_VALUES", "false").lower() == "true"

print("MQTT broker:", MQTT_BROKER_HOST, MQTT_BROKER_PORT)
print("MQTT topic:", MQTT_TOPIC)
print("facility_id mode:", FACILITY_ID_MODE)


paho-mqtt is not installed yet. Dry-run mode still works; install requirements.txt before real MQTT publishing.
MQTT broker: localhost 1883
MQTT topic: openelectricity/nem/facility_power_emissions
facility_id mode: unit_code


## 3.2 Message preparation and JSON schema

Clean values, load facility metadata when available, merge optional market data, sort records by event time, and convert each row into the JSON message sent to MQTT.

In [22]:
def _clean_string(value):
    """Return a trimmed string or None."""
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def _safe_float(value):
    """Convert numeric values to float while preserving missing values as None."""
    if pd.isna(value):
        return None
    return float(value)


def _iso_timestamp(value):
    """Return an ISO-8601 timestamp string with timezone information where available."""
    ts = pd.Timestamp(value)
    return ts.isoformat()


def build_unit_metadata_from_facilities(facilities):
    """
    Build a unit-to-facility lookup from OpenElectricity facilities_df.

    This works when this Task 3/6 section is appended to the first teammate's notebook,
    because their retrieval cells create facilities_df from the OpenElectricity facility endpoint.
    """
    rows = []

    if facilities is None or len(facilities) == 0:
        return pd.DataFrame()

    for _, facility in facilities.iterrows():
        facility_code = facility.get("code")
        facility_name = facility.get("name")
        network_region = facility.get("network_region")
        units = facility.get("units", [])

        if not isinstance(units, list):
            continue

        for unit in units:
            if not isinstance(unit, dict):
                continue

            rows.append({
                "unit_code": unit.get("code"),
                "facility_code": facility_code,
                "facility_name": facility_name,
                "network_region": network_region,
                "fueltech_id": unit.get("fueltech_id"),
                "status_id": unit.get("status_id")
            })

    metadata = pd.DataFrame(rows)
    if metadata.empty:
        return metadata

    metadata["unit_code"] = metadata["unit_code"].astype(str).str.strip()
    metadata = metadata.dropna(subset=["unit_code"]).drop_duplicates("unit_code")
    return metadata


def load_unit_metadata():
    """
    Load optional facility/unit metadata.

    Priority:
    1. facilities_df from the earlier OpenElectricity retrieval cells
    2. unit_metadata_for_dashboard.csv if it already exists
    3. empty metadata, with unit_code used as the safest fallback ID
    """
    if "facilities_df" in globals():
        metadata = build_unit_metadata_from_facilities(globals()["facilities_df"])
        if not metadata.empty:
            metadata.to_csv("unit_metadata_for_dashboard.csv", index=False)
            print("unit_metadata_for_dashboard.csv saved for the dashboard teammate.")
            return metadata

    metadata_path = Path("unit_metadata_for_dashboard.csv")
    if metadata_path.exists():
        metadata = pd.read_csv(metadata_path)
        metadata["unit_code"] = metadata["unit_code"].astype(str).str.strip()
        print("Loaded existing unit_metadata_for_dashboard.csv")
        return metadata

    print("No facilities_df metadata found. The publisher will use unit_code as facility_id.")
    return pd.DataFrame()


def load_market_data(path=MARKET_CSV):
    """Load optional market price/demand data if it is available."""
    if not Path(path).exists():
        return pd.DataFrame()

    market = pd.read_csv(path)
    required = {"region", "timestamp", "price", "demand"}
    if not required.issubset(market.columns):
        print("market_price_demand.csv exists but does not have the expected columns; skipping market merge.")
        return pd.DataFrame()

    market = market.copy()
    market["region"] = market["region"].astype(str).str.strip()
    market["timestamp"] = pd.to_datetime(market["timestamp"], errors="coerce")
    market["price"] = pd.to_numeric(market["price"], errors="coerce")
    market["demand"] = pd.to_numeric(market["demand"], errors="coerce")
    market = market.dropna(subset=["region", "timestamp"])
    market = market.drop_duplicates(subset=["region", "timestamp"])
    return market


def prepare_stream_dataframe(
    combined_path=COMBINED_CSV,
    market_path=MARKET_CSV,
    facility_id_mode=FACILITY_ID_MODE
):
    """
    Load, clean, enrich, and sort the Task 2 cached dataset for MQTT publishing.
    """
    if not Path(combined_path).exists():
        raise FileNotFoundError(
            f"{combined_path} not found. Run Task 1 and Task 2 first so the cached CSV is generated."
        )

    df = pd.read_csv(combined_path)
    required = {"unit_code", "timestamp", "power", "emissions"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{combined_path} is missing required columns: {sorted(missing)}")

    df = df.copy()
    df["unit_code"] = df["unit_code"].astype(str).str.strip()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["power"] = pd.to_numeric(df["power"], errors="coerce")
    df["emissions"] = pd.to_numeric(df["emissions"], errors="coerce")

    df = df.dropna(subset=["unit_code", "timestamp"])
    df = df.drop_duplicates(subset=["unit_code", "timestamp"], keep="last")

    metadata = load_unit_metadata()
    if not metadata.empty:
        df = df.merge(metadata, on="unit_code", how="left")
    else:
        df["facility_code"] = None
        df["facility_name"] = None
        df["network_region"] = None
        df["fueltech_id"] = None
        df["status_id"] = None

    # Set the ID used by the dashboard. The message also keeps both unit_code and facility_code.
    if facility_id_mode == "facility_code" and "facility_code" in df.columns:
        df["facility_id"] = df["facility_code"].where(df["facility_code"].notna(), df["unit_code"])
    else:
        df["facility_id"] = df["unit_code"]

    # Optional enrichment with market price and demand when region metadata exists.
    market = load_market_data(market_path)
    if not market.empty and "network_region" in df.columns:
        df = df.merge(
            market,
            left_on=["network_region", "timestamp"],
            right_on=["region", "timestamp"],
            how="left"
        )
    else:
        df["price"] = None
        df["demand"] = None

    df = df.sort_values(["timestamp", "unit_code"]).reset_index(drop=True)

    print(f"Prepared {len(df):,} stream records")
    print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"Unique unit_code values: {df['unit_code'].nunique():,}")
    print(f"Unique facility_id values: {df['facility_id'].nunique():,}")
    return df


def row_to_mqtt_message(row, sequence_number, replay_round):
    """
    Convert one cleaned dataframe row into the JSON schema consumed by Task 5.
    """
    message = {
        "schema_version": "1.0",
        "source": "openelectricity_cached_replay",
        "network": "NEM",
        "topic": MQTT_TOPIC,

        # Event-time order is based on this field.
        "timestamp": _iso_timestamp(row["timestamp"]),
        "event_time": _iso_timestamp(row["timestamp"]),
        "published_at": pd.Timestamp.utcnow().isoformat(),

        # ID fields for dashboard/database matching.
        "facility_id": _clean_string(row.get("facility_id")),
        "unit_code": _clean_string(row.get("unit_code")),
        "facility_code": _clean_string(row.get("facility_code")),
        "facility_name": _clean_string(row.get("facility_name")),

        # Optional display/filtering fields.
        "network_region": _clean_string(row.get("network_region")),
        "fuel_technology": _clean_string(row.get("fueltech_id")),
        "status": _clean_string(row.get("status_id")),

        # Measured values.
        "power_mw": _safe_float(row.get("power")),
        "emissions_tco2e": _safe_float(row.get("emissions")),

        # Optional market context; these are None when not available.
        "price_aud_per_mwh": _safe_float(row.get("price")),
        "demand_mw": _safe_float(row.get("demand")),

        # Debug/replay fields.
        "sequence_number": int(sequence_number),
        "replay_round": int(replay_round)
    }

    return message


## 3.3 MQTT client and one-pass publishing

Create the MQTT client and publish one chronological pass through the cached dataset. The `dry_run=True` option prints sample messages without connecting to a broker.

In [23]:
def create_mqtt_client():
    """Create and connect an MQTT client."""
    if mqtt is None:
        raise ImportError(
            "paho-mqtt is required for real MQTT publishing. "
            "Install it with: pip install paho-mqtt"
        )

    client_id = f"comp5339-task3-publisher-{uuid.uuid4().hex[:8]}"

    # paho-mqtt v2 supports CallbackAPIVersion; older versions do not.
    try:
        client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=client_id)
    except TypeError:
        client = mqtt.Client(client_id=client_id)

    if MQTT_USERNAME is not None:
        client.username_pw_set(MQTT_USERNAME, MQTT_PASSWORD)

    client.connect(MQTT_BROKER_HOST, MQTT_BROKER_PORT, keepalive=60)
    client.loop_start()
    print(f"Connected to MQTT broker as {client_id}")
    return client


def disconnect_mqtt_client(client):
    """Stop and disconnect the MQTT client safely."""
    if client is None:
        return
    client.loop_stop()
    client.disconnect()
    print("MQTT client disconnected.")


def _value_signature(message):
    """The fields used to decide whether a unit has changed since the last publication."""
    return (
        message.get("power_mw"),
        message.get("emissions_tco2e"),
        message.get("price_aud_per_mwh"),
        message.get("demand_mw")
    )


def publish_dataframe_once(
    df,
    client=None,
    topic=MQTT_TOPIC,
    replay_round=1,
    publish_delay_seconds=PUBLISH_DELAY_SECONDS,
    publish_only_changed_values=PUBLISH_ONLY_CHANGED_VALUES,
    dry_run=False,
    max_messages=None
):
    """
    Publish one full chronological pass through the dataframe.

    Set dry_run=True to print sample JSON messages without requiring a broker.
    """
    last_values_by_unit = {}
    published = 0
    skipped = 0

    for sequence_number, (_, row) in enumerate(df.iterrows(), start=1):
        message = row_to_mqtt_message(row, sequence_number, replay_round)

        if publish_only_changed_values:
            key = message["unit_code"]
            signature = _value_signature(message)
            if key in last_values_by_unit and last_values_by_unit[key] == signature:
                skipped += 1
                continue
            last_values_by_unit[key] = signature

        payload = json.dumps(message, ensure_ascii=False, separators=(",", ":"))

        if dry_run:
            if published < 5:
                print(payload)
        else:
            if mqtt is None:
                raise ImportError("paho-mqtt is required for real MQTT publishing.")
            if client is None:
                raise ValueError("client must be provided when dry_run=False")
            result = client.publish(topic, payload=payload, qos=0, retain=False)
            result.wait_for_publish()
            if result.rc != mqtt.MQTT_ERR_SUCCESS:
                raise RuntimeError(f"MQTT publish failed with rc={result.rc}")

        published += 1

        if published % 500 == 0:
            print(f"Published {published:,} messages in round {replay_round}")

        if max_messages is not None and published >= max_messages:
            break

        time.sleep(publish_delay_seconds)

    print(
        f"Round {replay_round} complete. Published={published:,}, "
        f"skipped_unchanged={skipped:,}, dry_run={dry_run}"
    )
    return published

In [24]:
# Task 3 quick validation: show one MQTT message without connecting to broker

stream_df = prepare_stream_dataframe()

sample_message = row_to_mqtt_message(
    stream_df.iloc[0],
    sequence_number=1,
    replay_round=1
)

print(json.dumps(sample_message, indent=2, ensure_ascii=False))

unit_metadata_for_dashboard.csv saved for the dashboard teammate.
Prepared 24,192 stream records
Time range: 2026-04-30 14:00:00+00:00 to 2026-05-07 13:55:00+00:00
Unique unit_code values: 13
Unique facility_id values: 13
{
  "schema_version": "1.0",
  "source": "openelectricity_cached_replay",
  "network": "NEM",
  "topic": "openelectricity/nem/facility_power_emissions",
  "timestamp": "2026-04-30T14:00:00+00:00",
  "event_time": "2026-04-30T14:00:00+00:00",
  "published_at": "2026-05-12T10:49:38.207326+00:00",
  "facility_id": "ADPBA1",
  "unit_code": "ADPBA1",
  "facility_code": "ADP",
  "facility_name": "Adelaide Desalination",
  "network_region": "SA1",
  "fuel_technology": "battery",
  "status": "operating",
  "power_mw": -0.01,
  "emissions_tco2e": 0.0,
  "price_aud_per_mwh": -2.01,
  "demand_mw": 1507.69,
  "sequence_number": 1,
  "replay_round": 1
}


# Task 6: Continuous Execution

Task 6 is implemented as a wrapper around Tasks 1–3. In this notebook it is placed immediately after Task 3 because it continuously replays the cached Task 2 data through the Task 3 MQTT publisher.

The continuous mode waits `60` seconds between replay rounds and keeps running until the notebook cell is stopped manually. For safe testing, the notebook also includes a limited dry-run mode.

## 6.1 Continuous publisher loop

Reload the cached CSV, publish records in chronological order, then wait 60 seconds before starting the next replay round.

In [25]:
def run_continuous_publisher(max_rounds=None,
    dry_run=False,
    max_messages_per_round=None
):
    """
    Continuously replay the cached Task 2 CSV as an ordered MQTT stream.

    For actual demonstration:
    - Set dry_run=False.
    - Make sure the broker is running.
    - Stop the notebook cell manually when finished.
    """
    client = None if dry_run else create_mqtt_client()
    replay_round = 0

    try:
        while True:
            replay_round += 1
            stream_df = prepare_stream_dataframe()

            publish_dataframe_once(
                stream_df,
                client=client,
                replay_round=replay_round,
                dry_run=dry_run,
                max_messages=max_messages_per_round
            )

            if max_rounds is not None and replay_round >= max_rounds:
                break

            print(f"Waiting {REPLAY_RESTART_DELAY_SECONDS} seconds before next replay round...")
            time.sleep(REPLAY_RESTART_DELAY_SECONDS)

    except KeyboardInterrupt:
        print("Publisher stopped by user.")
    finally:
        disconnect_mqtt_client(client)


## 6.2 Quick validation without MQTT

Run this cell first to check that the cached CSV can be loaded and that one JSON message has the expected schema.

In [26]:
# Quick validation: check the stream dataframe and message schema without connecting to MQTT.
stream_df = prepare_stream_dataframe()
print(stream_df.head())

sample_message = row_to_mqtt_message(stream_df.iloc[0], sequence_number=1, replay_round=1)
print(json.dumps(sample_message, indent=2, ensure_ascii=False))


unit_metadata_for_dashboard.csv saved for the dashboard teammate.
Prepared 24,192 stream records
Time range: 2026-04-30 14:00:00+00:00 to 2026-05-07 13:55:00+00:00
Unique unit_code values: 13
Unique facility_id values: 13
  unit_code                 timestamp  power  emissions facility_code  \
0    ADPBA1 2026-04-30 14:00:00+00:00  -0.01        0.0           ADP   
1   ADPBA1L 2026-04-30 14:00:00+00:00   0.01        0.0           ADP   
2    ADPPV1 2026-04-30 14:00:00+00:00   0.00        0.0           ADP   
3   ANGAST1 2026-04-30 14:00:00+00:00   0.00        0.0      ANGASTON   
4     ARWF1 2026-04-30 14:00:00+00:00  99.50        0.0          ARWF   

           facility_name network_region       fueltech_id  status_id  \
0  Adelaide Desalination            SA1           battery  operating   
1  Adelaide Desalination            SA1  battery_charging  operating   
2  Adelaide Desalination            SA1     solar_utility  operating   
3               Angaston            SA1        dist

## 6.3 Real continuous publisher switch

Keep `RUN_PUBLISHER = False` while editing or testing. Change it to `True` only when the MQTT broker and Task 5 dashboard are ready.

In [27]:
# Start the publisher when you are ready.
#
# Before running with RUN_PUBLISHER = True, give the dashboard teammate these values:
# - MQTT_BROKER_HOST
# - MQTT_BROKER_PORT
# - MQTT_TOPIC
# - whether username/password is needed
# - this JSON schema
# - whether facility_id should match unit_code or facility_code in the database
#
# For a safe test inside the notebook, keep RUN_PUBLISHER=False and run the dry-run below.

RUN_PUBLISHER = False

if RUN_PUBLISHER:
    # This will run continuously until you stop the cell.
    run_continuous_publisher(max_rounds=None, dry_run=False)
else:
    print("Publisher not started. Set RUN_PUBLISHER = True when the MQTT broker/dashboard is ready.")


Publisher not started. Set RUN_PUBLISHER = True when the MQTT broker/dashboard is ready.


## 6.4 Safe dry-run demonstration

This limited run prints sample messages and stops automatically. It proves the continuous execution code works without making the notebook run forever.

In [28]:
# Optional small test: print 10 messages instead of publishing them.
# This is useful before connecting to the dashboard.
run_continuous_publisher(max_rounds=1, dry_run=True, max_messages_per_round=10)


unit_metadata_for_dashboard.csv saved for the dashboard teammate.
Prepared 24,192 stream records
Time range: 2026-04-30 14:00:00+00:00 to 2026-05-07 13:55:00+00:00
Unique unit_code values: 13
Unique facility_id values: 13
{"schema_version":"1.0","source":"openelectricity_cached_replay","network":"NEM","topic":"openelectricity/nem/facility_power_emissions","timestamp":"2026-04-30T14:00:00+00:00","event_time":"2026-04-30T14:00:00+00:00","published_at":"2026-05-12T10:49:38.311849+00:00","facility_id":"ADPBA1","unit_code":"ADPBA1","facility_code":"ADP","facility_name":"Adelaide Desalination","network_region":"SA1","fuel_technology":"battery","status":"operating","power_mw":-0.01,"emissions_tco2e":0.0,"price_aud_per_mwh":-2.01,"demand_mw":1507.69,"sequence_number":1,"replay_round":1}
{"schema_version":"1.0","source":"openelectricity_cached_replay","network":"NEM","topic":"openelectricity/nem/facility_power_emissions","timestamp":"2026-04-30T14:00:00+00:00","event_time":"2026-04-30T14:00:00+